## Local embeddings modeling notebook

In [1]:
%load_ext autoreload
%autoreload 2

import geopandas as gpd
import duckdb
import os
from tqdm import tqdm


from datetime import datetime
import json
import os

import geopandas as gpd
import ipyleaflet as ipyl
from IPython.display import display
import ipywidgets as ipyw
import joblib
import numpy as np
import pandas as pd

import sys
import pathlib
sys.path.insert(0, 'src')

from embedding_store import get_annoy_index, from_duckdb
from ui import GeoLabeler, add_ee_basemaps


Error initializing Earth Engine: None could not be converted to bytes, defaulting to 


In [2]:
with open('config/ui_config.json', 'r') as f:
    config = json.load(f)

local_dir = config['local_dir']
annoy_index_path = os.path.join(local_dir, 'embeddings.ann')
# Load prebuilt Annoy index (build it once with the DuckDB embeddings pipeline, then reuse)
annoy_index = get_annoy_index(annoy_index_path, config['index_dim'])
tile_centroid_path = os.path.join(local_dir, 'centroid_gdf.parquet')
tile_centroid_gdf = gpd.read_parquet(tile_centroid_path)
duckdb_path = os.path.join(local_dir, 'embeddings.db')
embeddings_con = duckdb.connect(duckdb_path)
embeddings = from_duckdb(tile_centroid_gdf, embeddings_con, table_name='embeddings', id_column='tile_id')
valid_tile_dir = os.path.join(local_dir, 'tiles')

mgrs_ids = config['mgrs_ids']
start_date = config['start_date']
end_date = config['end_date']
imagery = config['imagery']

In [3]:
BOUNDARY_PATH = os.path.join(pathlib.Path().resolve(), "places/costa_rica.geojson")
BOUNDARY = gpd.read_file(BOUNDARY_PATH)

labeler = GeoLabeler(
    gdf=embeddings.gdf,
    geojson_path=BOUNDARY_PATH,
    save_dir=local_dir,
)
# Optional: add Earth Engine basemaps (requires EE auth). Comment to disable:
add_ee_basemaps(labeler, BOUNDARY_PATH, start_date, end_date)


label = ipyw.Label(); display(label)  

def handle_mouse_move(**kwargs):
    lat, lon = kwargs.get('coordinates')
    label_type = "Erase" if labeler.select_val == -100 else "Negative" if labeler.select_val == 0 else "Positive"
    label.value = f'Lat/lon: {lat:.4f}, {lon:.4f}. Mode: {"lasso" if labeler.lasso_mode else "single"}. Labeling: {label_type}'

labeler.map.on_interaction(handle_mouse_move)

Initializing GeoLabeler...
Adding controls...


Label(value='')

## Search
First search make take a while as the table is loaded into memory

In [9]:
# Get points
pos = embeddings.gdf.iloc[labeler.pos_indices]
neg = embeddings.gdf.iloc[labeler.neg_indices]

pos_vec = np.mean(embeddings.get_vectors(pos[embeddings.id_column]).values, axis=0)
if len(neg) > 0:
    neg_vec = np.mean(embeddings.get_vectors(neg[embeddings.id_column]).values, axis=0)
else:
    neg_vec = np.zeros(pos_vec.shape)

# Default query vector math, feel free to experiment with alternatives
query_vector = 2 * pos_vec - neg_vec


In [5]:
# Do ANN search

n_nbors = 13000

nbors = annoy_index.get_nns_by_vector(query_vector, n_nbors, include_distances=True)

# Filter out any IDs that are already in positive labels
nbors_filtered = [n for n in nbors[0] if n not in labeler.pos_indices]

detections = labeler.gdf.iloc[nbors_filtered]

# Update the GeoLabeler and map
labeler.detection_gdf = detections[['geometry']]
labeler.update_layer(
    labeler.points, json.loads(detections.geometry.to_json()))

## Export

In [12]:
# Export the positives and negatives
pos_export = labeler.gdf.iloc[labeler.pos_indices]
neg_export = labeler.gdf.iloc[labeler.neg_indices]

# Add label columns
pos_export['label'] = 1
neg_export['label'] = 0




In [ ]:
EXPORT_TYPE = "POSITIVE" # "FULL" or "POSITIVE"

if EXPORT_TYPE == "FULL":
# Combine into one gdf
    export_gdf = pd.concat([pos_export, neg_export], ignore_index=True)

elif EXPORT_TYPE == "POSITIVE":
    # Combine into one gdf
    export_gdf = pos_export

# Export to a parquet file
export_path = os.path.join(local_dir, f'{EXPORT_TYPE}_labels.parquet'.lower())
export_gdf.to_parquet(export_path, index=False)

## Classifer 
After export, please return to the README to follow the next steps through model training, inference, and post-processing.

# Load

## Option 1: Load a previously saved set of labels

In [5]:
# Helper function

def display_labels_on_labeler(labeler, labels_gdf, id_column):
    """
    Display positive and negative labels on the GeoLabeler instance.

    Args:
        labeler (GeoLabeler): The GeoLabeler instance.
        labels_gdf (GeoDataFrame): The GeoDataFrame containing labels and id column (e.g. tile_id).
        id_column (str): Name of the id column in labels_gdf and labeler.gdf (e.g. embeddings.id_column).
    """
    if labels_gdf is not None:
        pos_tile_ids = labels_gdf.loc[labels_gdf['label'] == 1, id_column].tolist()
        neg_tile_ids = labels_gdf.loc[labels_gdf['label'] == 0, id_column].tolist()

        # Get positions from labeler's GeoDataFrame where id is in pos/neg lists
        pos_indices = labeler.gdf[labeler.gdf[id_column].isin(pos_tile_ids)].index.tolist()
        neg_indices = labeler.gdf[labeler.gdf[id_column].isin(neg_tile_ids)].index.tolist()

        labeler.pos_indices = pos_indices
        labeler.neg_indices = neg_indices
        labeler.update_layers()
        print("Labels displayed on labeler.")
    else:
        print("No labels to display.")



In [6]:
# Load previously exported labels

labels_file_path = os.path.join(local_dir, 'full_labels.parquet')
if os.path.exists(labels_file_path):
    labels_gdf = gpd.read_parquet(labels_file_path)
    print(len(labels_gdf), "labels loaded")
    display_labels_on_labeler(labeler, labels_gdf, embeddings.id_column)

280 labels loaded
Labels displayed on labeler.


## Option 2: Load post-processed detections after model training and inference

In [7]:
# Add polygons from postprocess_detections.py

dissolved = gpd.read_parquet("/Users/ben/EarthGenome/data/costa_rica_pineapple/output/tile_classifier_predictions_1_costa_rica_posw1.0_prob_0.98_postprocess.parquet")

labeler.dissolve_layer = ipyl.GeoJSON(
    data=json.loads(dissolved.geometry.to_json()),
    style={'color': 'blue', 'opacity': 0.5, 'weight': 2, 'fillOpacity': 0.1},
    name='Dissolved Polygons'
)

labeler.map.add_layer(labeler.dissolve_layer)